# CLP: Lambda auf der Validation auswählen
Dieses Notebook führt unabhängige Trainingsläufe für mehrere CLP-Gewichte durch.
**Das Testset wird während der Suche nicht ausgewertet.** Modell, LoRA, Lernrate,
Tokenisierung und Split bleiben gleich. Nur Lambda und ggf. der Trainings-Seed variieren.

`lambda=0` verwendet denselben CLPTrainer/Collator, aber ohne Pairing-Loss und ohne
zusätzlichen Swap-Forward-Pass. Es ist die Referenz innerhalb dieser Suche.
Die voreingestellten Lambda-Werte sind ein erstes Suchraster, keine optimierten Werte.

Installation/Clone/Drive-Mount vorher wie gewohnt durchführen. Es werden keine
Trainingsläufe gestartet, bevor die ausdrücklich markierte Trainingszelle ausgeführt wird.

**Neue Auswahl:** Early Stopping ist aus; alle `EPOCHS` Epochen werden trainiert.
Alle Checkpoints bleiben erhalten und werden anschließend auf der Validation ausgewertet.
Für diese Version einen neuen Suchordner verwenden. Zusätzlicher Speicherbedarf: ein
vollständiger Checkpoint pro Epoche (inklusive Trainer-Zustand); zusätzliche Evaluation pro Checkpoint.

In [ ]:
from pathlib import Path
from datetime import datetime
import os, sys, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo = Path.cwd()
if not (repo / "src").is_dir():
    repo = repo.parent
if not (repo / "src").is_dir():
    raise RuntimeError("repo explizit auf den Projektordner setzen")
os.chdir(repo)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from src.config import ExperimentConfig, load_run_config
from src.experiments.run import run_experiment, resolve_run_config
from src.experiments.clp_search import rank_checkpoint_candidates


## Suchplan vor dem Training festlegen
Mit vier Lambdas und einem Seed entstehen vier Trainingsläufe. Für belastbarere
Vergleiche später dieselben zusätzlichen Seeds für jeden Kandidaten verwenden.
`MAX_F1_DROP=0.01` erlaubt einen absoluten Macro-F1-Rückgang von 0.01.
Diese Auswahlregel vor Betrachtung der Ergebnisse festlegen.

Die vorhandene LoRA-Config wird unverändert verwendet. Eine andere Auswahl von
`target_modules` wäre ein separates Experiment und sollte für alle Lambdas gleich sein.

In [ ]:
LAMBDA_VALUES = [0.0, 0.01, 0.1, 1.0]
SEEDS = [42]  # z.B. [42, 123, 2026]
MAX_F1_DROP = 0.03
EPOCHS = 8  # Feste Trainingsdauer für alle Lambdas und Seeds
MODEL_CONFIG = "configs/bert_lora.yaml"
DATASET_CONFIG = "configs/datasets/3D_combined.yaml"
RESULTS_ROOT = repo / "results"  # ggf. gemounteter Drive-Pfad

# Für Fortsetzung nach Kernel-Neustart hier den bestehenden Suchordner einsetzen.
SEARCH_DIR = RESULTS_ROOT / ("clp_search_" + datetime.now().strftime("%Y%m%d_%H%M%S"))

if 0.0 not in LAMBDA_VALUES or len(set(LAMBDA_VALUES)) != len(LAMBDA_VALUES):
    raise ValueError("Eindeutige Lambdas einschließlich 0.0 angeben")
if not all(np.isfinite(v) and v >= 0 for v in LAMBDA_VALUES):
    raise ValueError("Lambdas müssen endlich und nichtnegativ sein")
if not SEEDS or len(set(SEEDS)) != len(SEEDS):
    raise ValueError("Mindestens einen eindeutigen Seed angeben")

base_config = ExperimentConfig.from_files(
    model=MODEL_CONFIG,
    training="configs/training.yaml",
    evaluation="configs/evaluation.yaml",
    dataset=DATASET_CONFIG,
    mitigations=["configs/mitigation/clp.yaml"],
)

experiments = []
for seed in SEEDS:
    for weight in LAMBDA_VALUES:
        values = base_config.to_dict()
        values["training"]["seed"] = seed
        values["training"]["metric_for_best_model"] = "f1_macro"
        values["training"]["greater_is_better"] = True
        values["training"]["load_best_model_at_end"] = False
        values["training"]["early_stopping"] = {"enabled": False}
        values["training"]["epochs"] = EPOCHS
        values["training"]["save_total_limit"] = None  # Alle Epochen-Checkpoints behalten
        values["evaluation"].update(run_test=False, validation_fairness=False, counterfactual_testing=True, checkpoint_fairness=True)
        values["tokenization"] = {"max_length": 512, "truncation": True}
        values["mitigation"]["clp"].update(clp_lambda=weight, enabled=True)
        # Optional für geeignete GPU: values["training"]["fp16"] = True
        cfg = resolve_run_config(ExperimentConfig(values, list(base_config.sources)))
        key = hashlib.sha256(json.dumps(cfg.to_dict(), sort_keys=True).encode()).hexdigest()
        experiments.append({"seed": seed, "clp_lambda": weight, "key": key, "config": cfg})

display(pd.DataFrame([{k: v for k, v in e.items() if k != "config"} for e in experiments]))
print(f"{len(experiments)} Trainingsläufe; Ergebnisse: {SEARCH_DIR}")


## Training starten oder Suche fortsetzen
Die folgende Zelle startet echte Trainingsläufe. Jeder Lauf speichert Modell,
aufgelöste Config und Validation-Predictions. Abgeschlossene Läufe werden über ein
Manifest wiedergefunden; bei einem Fehler stoppt die Suche. Erneutes Ausführen
dieser Zelle setzt sie fort. Nach Kernel-Neustart denselben `SEARCH_DIR` setzen.
Ein geänderter Suchplan benötigt einen neuen Suchordner.

In [ ]:
SEARCH_DIR.mkdir(parents=True, exist_ok=True)
plan = {"selection_version": "lambda_epoch_v2", "keys": [e["key"] for e in experiments], "max_f1_drop": MAX_F1_DROP}
plan_path = SEARCH_DIR / "search_plan.json"
if plan_path.exists() and json.loads(plan_path.read_text()) != plan:
    raise ValueError("Suchplan geändert: neuen SEARCH_DIR verwenden")
plan_path.write_text(json.dumps(plan, indent=2), encoding="utf-8")
manifest_path = SEARCH_DIR / "completed_runs.json"
manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}

for experiment in experiments:
    key = experiment["key"]
    if key in manifest:
        run_dir = SEARCH_DIR / manifest[key]
        status = json.loads((run_dir / "status.json").read_text())
        saved = load_run_config(run_dir).to_dict()
        saved.pop("run", None)
        assert status["status"] == "completed"
        assert saved == experiment["config"].to_dict(), "Gespeicherte Config stimmt nicht überein"
        print("Bereits abgeschlossen:", run_dir)
        continue
    weight, seed = experiment["clp_lambda"], experiment["seed"]
    result = run_experiment(experiment["config"], f"lambda_{weight:g}_seed_{seed}", SEARCH_DIR)
    manifest[key] = Path(result["run_dir"]).relative_to(SEARCH_DIR).as_posix()
    temporary = manifest_path.with_suffix(".tmp")
    temporary.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    temporary.replace(manifest_path)


## Validation-Ergebnisse vergleichen
Die Counterfactual-Population besteht aus ursprünglich männlich/weiblich
zugeordneten Texten, die sich beim Swap ändern und nicht grammatisch unsicher sind.
Die Auswahl ist für alle Lambdas identisch; sie hängt nicht von den Vorhersagen ab.
Swaps außerhalb der Token-Grenze bleiben in dieser textbasierten Auswertung enthalten.

CTF Gap = Mittelwert von `abs(p_harmful(swapped) - p_harmful(original))`.
Flip Rate = Anteil geänderter vorhergesagter Klassen. Die Richtungen werden zusätzlich
getrennt ausgewiesen. Gruppenmetriken liegen je Checkpoint in
`checkpoint_evaluations/checkpoint-*/validation/normal/`.
Die hier berechneten Werte sind Punktschätzungen; Bootstrap kann später folgen.

In [ ]:
# Auch separat ohne erneutes Training ausführbar, wenn der Suchplan wieder aufgebaut wurde.
manifest = json.loads((SEARCH_DIR / "completed_runs.json").read_text())
rows = []
split_hashes = set()
for experiment in experiments:
    if experiment["key"] not in manifest:
        raise RuntimeError("Suche noch unvollständig; zuerst alle vorgesehenen Läufe abschließen")
    run_dir = SEARCH_DIR / manifest[experiment["key"]]
    reference = json.loads((run_dir / "split_reference.json").read_text())
    split_hashes.add(reference["assignments_sha256"])
    checkpoints = json.loads((run_dir / "checkpoint_metrics.json").read_text())
    if len(checkpoints) != EPOCHS or {round(r["epoch"], 6) for r in checkpoints} != set(range(1, EPOCHS + 1)):
        raise RuntimeError(f"Unvollständige Checkpoints: {run_dir}")
    for checkpoint in checkpoints:
        if not Path(checkpoint["checkpoint"]).is_dir():
            raise FileNotFoundError(checkpoint["checkpoint"])
        rows.append({"clp_lambda": experiment["clp_lambda"], "seed": experiment["seed"],
                     "run_dir": str(run_dir), **checkpoint})
assert len(split_hashes) == 1, "Unterschiedliche Dataset-Splits in derselben Suche"
results = pd.DataFrame(rows)
results.to_csv(SEARCH_DIR / "validation_results.csv", index=False)
display(results)

ranking = rank_checkpoint_candidates(rows, max_f1_drop=MAX_F1_DROP)
ranking.to_csv(SEARCH_DIR / "checkpoint_ranking.csv", index=False)
display(ranking)


## Trade-off visualisieren
Die Standardabweichung in der Tabelle beschreibt Schwankungen zwischen Seeds,
keine Konfidenzintervalle. Bei nur einem Seed ist sie nicht definiert.
Ein kleiner CTF Gap allein genügt nicht: Auch ein konstantes Modell könnte ihn erreichen.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for row in ranking.itertuples():
    color = "tab:blue" if row.eligible else "tab:gray"
    ax.scatter(row.ctf_mean, row.f1_mean, color=color)
    ax.annotate(f"λ={row.clp_lambda:g}, E={row.epoch:g}", (row.ctf_mean, row.f1_mean), xytext=(5, 5), textcoords="offset points")
ax.axhline(ranking["f1_threshold"].iloc[0], linestyle="--", color="tab:red", label="F1-Untergrenze")
ax.set(xlabel="Validation CTF Gap (kleiner ist besser)", ylabel="Validation Macro-F1 (größer ist besser)")
ax.legend()
fig.tight_layout()
fig.savefig(SEARCH_DIR / "validation_tradeoff.png", dpi=160)
plt.show()


## Lambda und Checkpoint gemeinsam auswählen
Unter allen Kombinationen aus Lambda und Epoche innerhalb der F1-Toleranz gewinnt
der kleinste mittlere CTF Gap. Gleichstände: höherer Macro-F1, kleineres Lambda, frühere Epoche.
Die Referenz ist der höchste mittlere Macro-F1 über die Epochen bei Lambda 0.
Diese eine F1-Untergrenze gilt für alle Kandidaten. Auch Lambda 0 kann gewinnen.
Bei mehreren Seeds wird eine gemeinsame Epoche anhand der Mittelwerte gewählt.
Individuelle Seeds können dabei unter der Grenze liegen: Die Bedingung gilt für den Mittelwert.

Die ausgewählten **Checkpoint-Pfade** aller Seeds werden referenziert.
`run_dir/model` enthält hier die letzte Epoche, nicht automatisch den ausgewählten Checkpoint. Das Notebook führt keine
Testauswertung und kein erneutes Training aus. Nach Festlegung der Auswahl können
diese gespeicherten Modelle auf dem Testset bewertet werden.

In [ ]:
best = ranking.iloc[0]
selected_lambda = float(best["clp_lambda"])
selected_epoch = float(best["epoch"])
selected_runs = results.loc[
    results["clp_lambda"].eq(selected_lambda) & results["epoch"].eq(selected_epoch),
    ["seed", "run_dir", "checkpoint", "evaluation_dir"],
]
selection = {
    "selected_lambda": selected_lambda,
    "selected_epoch": selected_epoch,
    "baseline_f1_reference": float(best["baseline_f1_reference"]),
    "f1_threshold": float(best["f1_threshold"]),
    "rule": "min mean validation CTF over (lambda, epoch), subject to mean F1 >= max_epoch mean_seed F1(lambda0) - max_f1_drop",
    "max_f1_drop": MAX_F1_DROP,
    "runs": selected_runs.to_dict(orient="records"),
}
(SEARCH_DIR / "selection.json").write_text(json.dumps(selection, indent=2), encoding="utf-8")
print("Ausgewähltes Lambda / Epoche:", selected_lambda, selected_epoch)
display(selected_runs)
